# PharmaPath: Exploratory Data Analysis (EDA)
This notebook explores the raw and processed data assets underpinning the PharmaPath network-pharmacology pipeline:
1. STRING v12 Homo sapiens PPI Graph and ID mapping fidelity
2. Crated drug target sets (Metformin & Imatinib)
3. LINCS L1000 chemical perturbation signatures
4. Open Targets Platform disease gene modules (Type 2 Diabetes & Chronic Myeloid Leukemia)


In [ ]:
import sys
from pathlib import Path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR / "src"))

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")


## 1. STRING v12 Interactome Scaffold Analysis
Inspect the Largest Connected Component (LCC) extracted from high-confidence (`combined_score >= 700`) STRING v12 edges.


In [ ]:
edges_df = pd.read_parquet(ROOT_DIR / "data" / "processed" / "ppi_edges.parquet")
print(f"Total Undirected High-Confidence Edges: {len(edges_df):,}")
print(edges_df.head())


In [ ]:
# Degree Distribution of the Human PPI Scaffold
G = nx.read_graphml(ROOT_DIR / "data" / "processed" / "ppi_lcc.graphml")
degrees = [d for n, d in G.degree()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(degrees, bins=50, color="#2980B9", edgecolor="black", log=True)
ax1.set_title("Degree Distribution (Log Scale)")
ax1.set_xlabel("Node Degree (k)")
ax1.set_ylabel("Count (log)")

# Log-Log scale to assess scale-free power law
sns.histplot(degrees, log_scale=(True, True), ax=ax2, color="#E67E22")
ax2.set_title("Log-Log Scale-Free Interactome Property")
ax2.set_xlabel("log(k)")
ax2.set_ylabel("log(P(k))")
plt.tight_layout()
plt.show()

print(f"Nodes: {G.number_of_nodes():,}, Edges: {G.number_of_edges():,}, Density: {nx.density(G):.6f}")


## 2. Drug Targets Exploration
Compare the curated targets for Metformin (mitochondrial Complex I + AMPK subunits) and Imatinib (tyrosine kinases).


In [ ]:
targets_met = pd.read_parquet(ROOT_DIR / "data" / "processed" / "targets_metformin.parquet")
targets_ima = pd.read_parquet(ROOT_DIR / "data" / "processed" / "targets_imatinib.parquet")

print("=== METFORMIN TARGETS ===")
print(f"Total: {len(targets_met)}, In PPI LCC: {targets_met['in_ppi'].sum()}")
print(targets_met[['gene_symbol', 'in_ppi', 'details']].head(10))

print("\n=== IMATINIB TARGETS ===")
print(f"Total: {len(targets_ima)}, In PPI LCC: {targets_ima['in_ppi'].sum()}")
print(targets_ima[['gene_symbol', 'in_ppi', 'details']])


## 3. Open Targets Disease Modules
Inspect disease gene sets retrieved via Open Targets GraphQL API (Score >= 0.2).


In [ ]:
t2d_df = pd.read_parquet(ROOT_DIR / "data" / "raw" / "opentargets_type_2_diabetes.parquet")
cml_df = pd.read_parquet(ROOT_DIR / "data" / "raw" / "opentargets_chronic_myeloid_leukemia.parquet")

print(f"Type 2 Diabetes Genes: {len(t2d_df)}, Top 5:")
print(t2d_df.head(5))

print(f"\nChronic Myeloid Leukemia Genes: {len(cml_df)}, Top 5:")
print(cml_df.head(5))


## 4. LINCS L1000 Expression Signatures
Explore chemically perturbed gene expression signatures derived via Enrichr.


In [ ]:
lincs_met = pd.read_parquet(ROOT_DIR / "data" / "raw" / "lincs_metformin.parquet")
lincs_ima = pd.read_parquet(ROOT_DIR / "data" / "raw" / "lincs_imatinib.parquet")

print(f"Metformin LINCS Perturbed Genes: {len(lincs_met)}")
print(lincs_met.head())

print(f"\nImatinib LINCS Perturbed Genes: {len(lincs_ima)}")
print(lincs_ima.head())
